In [1]:
import pandas as pd
import numpy as np
import os
import json

In [2]:
## print the working directory
print("Current Working Directory: ", os.getcwd())

Current Working Directory:  /Users/jillmarley/Scorecard-Metrics-Over-Time/local_metrics


In [3]:
package_data = pd.read_csv("../package_data_final.csv")

In [8]:
## count the number of tag_commit_sha per github_repos 
## we can use this to verify we collected all of the data running scorecard locally

tag_commit_counts_package_data = package_data.groupby('github_repo')['tag_commit_sha'].nunique().reset_index()
tag_commit_counts_package_data.columns = ['github_repo', 'tag_commit_count']

In [6]:
import pandas as pd
import json
import os
from pathlib import Path

# Define the base directory
base_dir = Path('data')

# Create lists to store flattened data from all files
all_records = []

# Iterate through each data_run folder
for folder_num in range(1, 16):
    folder_name = f'data_run{folder_num}'
    folder_path = base_dir / folder_name
    
    print(f"Processing {folder_name}...")
    
    # Check if folder exists
    if not folder_path.exists():
        print(f"  Warning: {folder_name} not found, skipping...")
        continue
    
    # Get all JSON files in this folder
    json_files = list(folder_path.glob('*.json'))
    print(f"  Found {len(json_files)} JSON files")
    
    # Process each JSON file
    for json_file in json_files:
        try:
            # Load the JSON data
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            # Process each entry in the JSON file
            for entry in data:
                # Base record with metadata
                record = {
                    #'data_run_folder': folder_name,  # Track which folder this came from
                    'source_file': json_file.name,    # Track which file this came from
                    'github_repo': entry['github_repo'],
                    'tag_name': entry['tag_name'],
                    'tag_commit_sha': entry['tag_commit_sha'],
                    'published_at': entry['published_at'],
                    'project_name': entry['project_name'],
                    'package_name': entry['package_name'],
                    'run_timestamp': entry['run_timestamp'],
                    'aggregate_score': entry['score']
                }
                
                # Add check-specific columns
                for check in entry['checks']:
                    check_name = check['name']
                    record[f'{check_name}_score'] = check['score']
                    record[f'{check_name}_reason'] = check['reason']
                
                all_records.append(record)
        
        except json.JSONDecodeError as e:
            print(f"  Error reading {json_file.name}: {e}")
        except KeyError as e:
            print(f"  Missing expected key in {json_file.name}: {e}")
        except Exception as e:
            print(f"  Unexpected error with {json_file.name}: {e}")

# Create DataFrame from all records
df = pd.DataFrame(all_records)

# Display summary
print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"Total records collected: {len(all_records)}")
print(f"DataFrame shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumns: {df.columns.tolist()}")

# Check for duplicates (optional but recommended)
duplicates = df.duplicated(subset=['github_repo', 'tag_name', 'tag_commit_sha']).sum()
print(f"\nDuplicate entries found: {duplicates}")

# Drop duplicates from dataframe
df = df.drop_duplicates(subset=['github_repo', 'tag_name', 'tag_commit_sha'])

# Save to CSV
output_file = 'scorecard_local_metrics_jill_updated.csv'
df.to_csv(output_file, index=False)
print(f"\nData saved to: {output_file}")

Processing data_run1...
  Found 1090 JSON files
Processing data_run2...
  Found 585 JSON files
Processing data_run3...
  Found 25 JSON files
Processing data_run4...
  Found 1423 JSON files
Processing data_run5...
  Found 41 JSON files
Processing data_run6...
  Found 453 JSON files
Processing data_run7...
  Found 737 JSON files
Processing data_run8...
  Found 1979 JSON files
Processing data_run9...
  Found 122 JSON files
Processing data_run10...
  Found 297 JSON files
Processing data_run11...
  Found 464 JSON files
Processing data_run12...
  Found 1303 JSON files
Processing data_run13...
  Found 14 JSON files
Processing data_run14...
  Found 1968 JSON files
Processing data_run15...
  Found 922 JSON files

Processing complete!
Total records collected: 228958
DataFrame shape: (228958, 27)

First few rows:
                                         source_file  \
0  adafruit_adafruit_circuitpython_framebuf__adaf...   
1  adafruit_adafruit_circuitpython_framebuf__adaf...   
2  adafruit_adafru

## OLD - Comparison Analysis

In [12]:
tag_commit_counts_local_check = df.groupby('github_repo')['tag_commit_sha'].nunique().reset_index()
tag_commit_counts_local_check.columns = ['github_repo', 'tag_commit_count']

In [13]:
merged = pd.merge(
    tag_commit_counts_local_check[['github_repo', 'tag_commit_count']], 
    tag_commit_counts_package_data[['github_repo', 'tag_commit_count']], 
    on=['github_repo'],
    how='inner',
    suffixes=('_local', '_package'),
    indicator=True
)

# Add a column to flag differences
merged['counts_match'] = merged['tag_commit_count_local'] == merged['tag_commit_count_package']
merged['difference'] = merged['tag_commit_count_local'] - merged['tag_commit_count_package']

# Display results
print(f"Total repos in both datasets: {len(merged)}")
print(f"\nRepos with MATCHING counts: {merged['counts_match'].sum()}")
print(f"Repos with DIFFERENT counts: {(~merged['counts_match']).sum()}")

# Show repos with differences
merged.to_csv('did_local_checks_capture_all_tag_shas.csv', index=False)


Total repos in both datasets: 8148

Repos with MATCHING counts: 7678
Repos with DIFFERENT counts: 470


In [20]:
print(merged[merged['counts_match'] == False]['github_repo'])
missing_commits = (merged[merged['counts_match'] == False]['github_repo']).tolist()

2                       https://github.com/0x-jerry/utils
5                   https://github.com/0xs34n/starknet.js
79              https://github.com/abichinger/vue-js-cron
90               https://github.com/abstra-app/abstra-lib
268                 https://github.com/admiralds/react-ui
                              ...                        
6457              https://github.com/keras-team/keras-nlp
6481                https://github.com/khan/wonder-blocks
6502                https://github.com/kikobeats/html-get
6560    https://github.com/kmagiera/react-native-gestu...
6627            https://github.com/koenkk/zigbee-herdsman
Name: github_repo, Length: 98, dtype: object


In [22]:
missing_dataset = package_data[package_data['github_repo'].isin(missing_commits)]
missing_dataset.to_csv('missing_tag_commits_in_local_checks.csv', index=False)

## Comparison Analysis

In [13]:
local_checks = pd.read_csv('scorecard_local_metrics.csv')

**NOTE that Tanmay has half of the local check data that is not located in the scorecard_flattened_data**

## We added about 1700 repositories on January 1, 2026. We need to collect the local metric data for these repositories. 

In [18]:
OLD_package_data = pd.read_csv("../OLD/ORIGINAL_package_data.csv")

In [20]:
current_package_data = pd.read_csv("../package_data_final.csv")

In [21]:
# Find repositories in package_data that are NOT in old_package_data
new_repos = current_package_data[~current_package_data['github_repo'].isin(OLD_package_data['github_repo'])]

In [34]:
# First, ensure both dataframes have the same columns
common_cols = list(set(package_data.columns) & set(old_package_data.columns))

# Find rows that are NOT in old_package_data
new_rows = package_data.merge(
    old_package_data[common_cols],
    on=common_cols,
    how='left',
    indicator=True
)
new_rows = new_rows[new_rows['_merge'] == 'left_only'].drop(columns='_merge')

print(f"Found {len(new_rows)} new rows")
print(f"Difference: {len(package_data)} - {len(old_package_data)} = {len(package_data) - len(old_package_data)}")

Found 97893 new rows
Difference: 345394 - 250709 = 94685


In [36]:
new_rows.to_csv('new_data_to_get_local_checks_for.csv', index=False)

## Tanmay's Data

In [5]:
import pandas as pd
import json
from pathlib import Path

# Create lists to store flattened data from all files
all_records = []

folder_path = Path("tanmay_local_checks/final_data/")
json_files = list(folder_path.glob('*.json'))
print(f"Found {len(json_files)} JSON files")

for json_file in json_files:
    with open(json_file, 'r') as f:
        try:
            data = json.load(f)
            for entry in data:
                try:
                    # Use .get() for optional fields to prevent the crash
                    record = {
                        'source_file': json_file.name,
                        'github_repo': entry['github_repo'],
                        'tag_name': entry.get('tag_name', 'MISSING_TAG'), 
                        'tag_commit_sha': entry['tag_commit_sha'],
                        'published_at': entry.get('published_at', 'N/A'),
                        'project_name': entry['project_name'],
                        'package_name': entry['package_name'],
                        'run_timestamp': entry['run_timestamp'],
                        'aggregate_score': entry['score']
                    }
                    
                    for check in entry['checks']:
                        check_name = check['name']
                        record[f'{check_name}_score'] = check['score']
                        record[f'{check_name}_reason'] = check['reason']
                    
                    all_records.append(record)

                except KeyError as e:
                    print(f"  Missing key {e} in an entry within {json_file.name}")
        
        except json.JSONDecodeError:
            print(f"  Failed to parse {json_file.name}")

# Create DataFrame
df = pd.DataFrame(all_records)

# Display summary
print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"Total records collected: {len(all_records)}")
print(f"DataFrame shape: {df.shape}")

# Save to CSV
output_file = 'scorecard_local_metrics_tanmay.csv'
df.to_csv(output_file, index=False)
print(f"Data saved to: {output_file}")

Found 6837 JSON files

Processing complete!
Total records collected: 121062
DataFrame shape: (121062, 27)
Data saved to: scorecard_local_metrics_tanmay.csv


## Manually copying and paste the datasets into one file: scorecard_local_metrics.csv

## Did the local script collect data on all package tags pairs? 

**It makes more sense to check after we have all of the data.** We will have to remove some of the data we collected local metrics for, as we removed some of the packages due to the revised inclusion / exclusion criteria.

In [2]:
import pandas as pd
package_data = pd.read_csv("../package_data.csv")
local_checks = pd.read_csv('scorecard_local_metrics_jill.csv')

In [18]:
## subtract the repositories in new_data_to_get_local_checks_for.csv from package_data
new_data = pd.read_csv('new_data_to_get_local_checks_for.csv')
new_repos = new_data['github_repo'].unique().tolist()
new_package_data = package_data[package_data['githubrepo'].isin(new_repos)]
package_data_we_should_have_data_for = package_data[~package_data['githubrepo'].isin(new_repos)]

In [4]:
import pandas as pd
data1 = pd.read_csv('scorecard_local_metrics_jill.csv')
data2 = pd.read_csv('scorecard_local_metrics_tanmay.csv') 
data = pd.concat([data1, data2])

data.to_csv("scorecard_local_metrics_13k.csv", index=False)

In [6]:
import pandas as pd
local_data = pd.read_csv("../scorecard_local_metrics_13k.csv")

## Adding in new runs of local metric data

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path

# Define the base directory
base_dir = Path('data')

# Create lists to store flattened data from all files
all_records = []

# Iterate through each data_run folder
for folder_num in range(1, 14):
    folder_name = f'data_run{folder_num}'
    folder_path = base_dir / folder_name
    
    print(f"Processing {folder_name}...")
    
    # Check if folder exists
    if not folder_path.exists():
        print(f"  Warning: {folder_name} not found, skipping...")
        continue
    
    # Get all JSON files in this folder
    json_files = list(folder_path.glob('*.json'))
    print(f"  Found {len(json_files)} JSON files")
    
    # Process each JSON file
    for json_file in json_files:
        try:
            # Load the JSON data
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            # Process each entry in the JSON file
            for entry in data:
                # Base record with metadata
                record = {
                    #'data_run_folder': folder_name,  # Track which folder this came from
                    'source_file': json_file.name,    # Track which file this came from
                    'github_repo': entry['github_repo'],
                    'tag_name': entry['tag_name'],
                    'tag_commit_sha': entry['tag_commit_sha'],
                    'published_at': entry['published_at'],
                    'project_name': entry['project_name'],
                    'package_name': entry['package_name'],
                    'run_timestamp': entry['run_timestamp'],
                    'aggregate_score': entry['score']
                }
                
                # Add check-specific columns
                for check in entry['checks']:
                    check_name = check['name']
                    record[f'{check_name}_score'] = check['score']
                    record[f'{check_name}_reason'] = check['reason']
                
                all_records.append(record)
        
        except json.JSONDecodeError as e:
            print(f"  Error reading {json_file.name}: {e}")
        except KeyError as e:
            print(f"  Missing expected key in {json_file.name}: {e}")
        except Exception as e:
            print(f"  Unexpected error with {json_file.name}: {e}")

# Create DataFrame from all records
df = pd.DataFrame(all_records)

# Display summary
print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"Total records collected: {len(all_records)}")
print(f"DataFrame shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumns: {df.columns.tolist()}")

# Check for duplicates (optional but recommended)
duplicates = df.duplicated(subset=['github_repo', 'tag_name', 'tag_commit_sha']).sum()
print(f"\nDuplicate entries found: {duplicates}")

# Save to CSV
output_file = 'scorecard_local_metrics_jill_updated.csv'
df.to_csv(output_file, index=False)
print(f"\nData saved to: {output_file}")

## **January 30 Updates:** Combining (hopefully) all of the remaining Scorecard data?

1. Manually combine scorecard_local_metrics_jill_updated.csv with scorecard_local_metrics_tanmay.csv to create scorecard_local_metrics_updated.csv
2. Removed any repositories not found in package_data.csv

In [ ]:
## first need to check that our data formats are the same 

tanmay = pd.read_csv('scorecard_local_metrics_tanmay.csv')
jill = pd.read_csv('scorecard_local_metrics_jill_updated.csv')

total = pd.concat([tanmay, jill])

In [15]:
## how many duplicates are there?
duplicates = total.duplicated(subset=['github_repo', 'tag_name', 'tag_commit_sha']).sum()
print(f"\nDuplicate entries found: {duplicates}")

## remove the duplicates    
total_removed_dups = total.drop_duplicates(subset=['github_repo', 'tag_name', 'tag_commit_sha'])

total_removed_dups.to_csv('scorecard_local_metrics_updated.csv', index=False)


Duplicate entries found: 152172


## What data are we missing rows for?

In [ ]:
## there are repositories in scorecard_local_metrics that are not in package_data and need to be removed from scorecard_local_metrics
# local_metrics = pd.read_csv("scorecard_local_metrics_updated.csv")
# package_data = pd.read_csv("../package_data.csv")

# ## which repositories exist in local_metrics but not in package_data? 
# missing_in_package_data = local_metrics[~local_metrics[['github_repo']].isin(package_data[['githubrepo']])]

# local_metrics_cleaned = local_metrics[local_metrics["github_repo"].isin(package_data["githubrepo"])]
# local_metrics_cleaned = local_metrics_cleaned.drop(columns = 'Unnamed: 0')

# local_metrics_cleaned.to_csv("scorecard_local_metrics_updated.csv", index = False)


In [54]:
## extract the vulnerability count and save it to the csv file 

local_metrics = pd.read_csv("scorecard_local_metrics_updated.csv")
local_metrics["Vulnerability_count"] = (
    local_metrics["Vulnerabilities_reason"]
    .str.extract(r"^(\d+)")
    .astype(int)
)


local_metrics.to_csv("scorecard_local_metrics_updated.csv", index = False)

## The repositories that missing in package_data are now removed from the scorecard local data.

In [51]:
local_metrics = pd.read_csv("scorecard_local_metrics_updated.csv")
package_data = pd.read_csv("../package_data.csv")

In [50]:
## need to determine which rows are missing from local_metrics that are in package_data

package_data_dedup = package_data.drop_duplicates(subset=['githubrepo', 'tag_name', 'tag_commit_sha'])

missing_in_local_metrics = package_data_dedup[
    ~package_data_dedup[['githubrepo', 'tag_commit_sha']].apply(tuple, axis=1).isin(
        local_metrics[['github_repo', 'tag_commit_sha']].apply(tuple, axis=1)
    )
]

## grouping by repository name, which tag names correspond to the same commit sha? 
# grouped = (
#     missing_in_local_metrics_tagName
#     .groupby(["githubrepo", "tag_commit_sha"])["tag_name"]
#     .apply(list)
#     .reset_index()
# )

## it seems like the same commit corresponds to multiple tag names

missing_in_local_metrics.to_csv("missing_data_rerun.csv")